In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Step 1: Load the dataset into a pandas DataFrame
df = pd.read_csv('HDHI Admission data.csv')

# Step 2: Drop the first four rows
df = df.iloc[4:]

# Step 3: Drop rows that have the value "EMPTY"
df = df.replace("EMPTY", pd.NA)  # Replace "EMPTY" with NaN (Not a Number)
df.dropna(inplace=True)  # Drop rows where any NaN value exists

# Step 4: Drop the first four columns
df = df.iloc[:, 4:]

# Step 5: Identify numeric columns
numeric_cols = ['AGE', 'DURATION OF STAY', 'duration of intensive unit stay', 
                'HB', 'TLC', 'PLATELETS', 'GLUCOSE', 'UREA', 'CREATININE', 
                'BNP', 'RAISED CARDIAC ENZYMES', 'EF']

# Identify categorical columns
categorical_cols = [col for col in df.columns if col not in numeric_cols]

# Step 6: Convert categorical features to numerical (using one-hot encoding)
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Step 7: Split the data into features (X) and target (y)
X = df.drop('DURATION OF STAY', axis=1)
y = df['DURATION OF STAY']

# Step 8: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 9: Initialize the Random Forest Regressor
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Step 10: Train the model
rf_regressor.fit(X_train, y_train)

# Step 11: Make predictions on the test set
y_pred = rf_regressor.predict(X_test)

# Step 12: Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f'Mean Absolute Error: {mae:.2f}')
print(f'Root Mean Squared Error: {rmse:.2f}')


Mean Absolute Error: 1.91
Root Mean Squared Error: 3.11


In [17]:
example_data = pd.DataFrame({
    'AGE': [65],
    'duration of intensive unit stay': [3],
    'HB': [13.5],
    'TLC': [9.5],
    'PLATELETS': [250],
    'GLUCOSE': [110],
    'UREA': [30],
    'CREATININE': [1.2],
    'BNP': [1800],
    'RAISED CARDIAC ENZYMES': [1],
    'EF': [40],
    'GENDER_M': [1],
    'RURAL_U': [1],
    'TYPE OF ADMISSION_EMERGENCY': [1],
    'OUTCOME_DISCHARGE': [1]
})

# Ensure example_data has the same columns as X_train
missing_cols = set(X_train.columns) - set(example_data.columns)
for col in missing_cols:
    example_data[col] = 0
example_data = example_data[X_train.columns]

# Step 14: Predict the Duration of Stay for the example data
predicted_duration = rf_regressor.predict(example_data)
print(f'Predicted Duration of Stay: {predicted_duration[0]:.2f} days')

Predicted Duration of Stay: 5.24 days


In [18]:
from sklearn.model_selection import GridSearchCV

# Step 1: Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

# Step 2: Initialize the GridSearchCV with the RandomForestRegressor and the parameter grid
grid_search = GridSearchCV(estimator=rf_regressor, param_grid=param_grid, 
                           cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Step 3: Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Step 4: Get the best parameters and the best estimator
best_params = grid_search.best_params_
best_rf = grid_search.best_estimator_

# Step 5: Make predictions on the test set using the best model
y_pred_best = best_rf.predict(X_test)

# Step 6: Evaluate the tuned model
mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = mean_squared_error(y_test, y_pred_best, squared=False)

print(f'Best Parameters: {best_params}')
print(f'Mean Absolute Error (Best Model): {mae_best:.2f}')
print(f'Root Mean Squared Error (Best Model): {rmse_best:.2f}')



Fitting 3 folds for each of 324 candidates, totalling 972 fits


C:\Users\almas\anaconda3\Lib\site-packages\sklearn\ensemble\_forest.py:413: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(


Best Parameters: {'max_depth': None, 'max_features': 'auto', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 300}
Mean Absolute Error (Best Model): 1.90
Root Mean Squared Error (Best Model): 3.07


In [20]:
from sklearn.model_selection import GridSearchCV

# Refined parameter grid with a broader range
param_grid = {
    'n_estimators': [300, 500, 700],  # Increase the number of trees
    'max_depth': [None, 10, 20, 30, 40],  # Explore more depth options
    'min_samples_split': [2, 5, 10],  # Test smaller splits
    'min_samples_leaf': [1, 2, 4],  # Smaller leaf sizes
    'max_features': [1.0, 'sqrt', 'log2'],  # Use recommended max_features values
}

# Initialize the Random Forest Regressor
rf_regressor = RandomForestRegressor(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=rf_regressor, param_grid=param_grid, 
                           cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and the best estimator
best_params = grid_search.best_params_
best_rf = grid_search.best_estimator_

# Make predictions on the test set using the best model
y_pred_best = best_rf.predict(X_test)

# Evaluate the tuned model
mae_best = mean_absolute_error(y_test, y_pred_best)
rmse_best = mean_squared_error(y_test, y_pred_best, squared=False)

print(f'Best Parameters: {best_params}')
print(f'Mean Absolute Error (Best Model): {mae_best:.2f}')
print(f'Root Mean Squared Error (Best Model): {rmse_best:.2f}')


Fitting 3 folds for each of 405 candidates, totalling 1215 fits
Best Parameters: {'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 300}
Mean Absolute Error (Best Model): 1.90
Root Mean Squared Error (Best Model): 3.07
